# 01 - JOB-01: official data access and B2 (gated: acquire_data)

**Private compute session - not a dataset mirror.** Raw BraTS data come only from the official TCIA source into ephemeral session storage (`WORK`) and are never written to notebook output, a Kaggle Dataset, GitHub or the website. B1 basis: owner-approved alternative (amendment v1.0-A1); no TCIA authorization is claimed. Enable **GPU** and **Internet** in the notebook settings. Never paste credentials into a cell; use Kaggle Secrets if a private repository needs a token.

In [ ]:
# PARAMETERS - set before running (no secrets here)
REPO_URL = ""   # https URL of this research repository (GitHub)
COMMIT = ""     # exact commit SHA or tag to run (never a moving branch)
WORK = "/tmp/brats"                     # ephemeral session storage (raw data stay here)
EXPORT = "/kaggle/working/export"       # records/reports/artifacts only - never raw data
assert REPO_URL, "set REPO_URL"
assert COMMIT, "set COMMIT"


In [ ]:
import os
import subprocess

os.makedirs(WORK, exist_ok=True)
os.makedirs(EXPORT, exist_ok=True)
!git clone --quiet {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout --quiet {COMMIT}
!git rev-parse HEAD
!pip install --quiet -e ".[io,nnunet]"
!brats-uncertainty verify-protocol

In [ ]:
r = subprocess.run(["brats-uncertainty", "check-action", "acquire_data"], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, "gate not open for acquire_data: stop here"

In [ ]:
!brats-uncertainty compute-preflight --job JOB-01 --work-dir {WORK} --json {EXPORT}/probe_JOB-01.json

## Storage preflight
Enter the selection size the transfer client reports for `BraTS2021_TrainingSet` + `RSNA-ASNR-MICCAI-BraTS-2021.sums`. Never guess. Continue only on `PREFLIGHT OK`.

In [ ]:
SELECTED_GIB = None  # from the transfer client listing
assert SELECTED_GIB, "enter the measured selection size"
DELIVERY = f"{WORK}/delivery"
!brats-uncertainty storage-preflight --selected-gib {SELECTED_GIB} --delivery-dir {DELIVERY} --storage-dir {WORK}/raw

## Official delivery
Source: the **DOWNLOAD (142GB)** public package on https://www.cancerimagingarchive.net/analysis-result/rsna-asnr-miccai-brats-2021/ (TCIA Faspex). Select ONLY `RSNA-ASNR-MICCAI-BraTS-2021.sums` and `RSNA-ASNR-MICCAI-BraTS-2021/BraTS2021_TrainingSet`; keep the hierarchy exactly as delivered.

- **A. Runtime acquisition:** set `RECEIVE_CMD` to the exact command confirmed by notebook 00.
- **B. Manual official delivery (private VM):** leave `RECEIVE_CMD` empty; the owner's download must already be in `DELIVERY`. (Not possible on Kaggle: uploading data would create a dataset.)

In [ ]:
RECEIVE_CMD = ""  # A: confirmed command; B: leave empty
os.makedirs(DELIVERY, exist_ok=True)
if RECEIVE_CMD:
    !{RECEIVE_CMD}
pkg = os.path.join(DELIVERY, "RSNA-ASNR-MICCAI-BraTS-2021", "BraTS2021_TrainingSet")
assert os.path.isdir(pkg), "official delivery not found in DELIVERY"

In [ ]:
# The two official B3/B4 files, saved byte-for-byte (never opened or re-saved)
import urllib.request

for name, url in [
    ("BraTS2021_MappingToTCIA.xlsx", "https://www.cancerimagingarchive.net/wp-content/uploads/BraTS2021_MappingToTCIA.xlsx"),
    ("UCSF-PDGM-metadata_v5.csv", "https://www.cancerimagingarchive.net/wp-content/uploads/UCSF-PDGM-metadata_v5.csv"),
]:
    if not os.path.exists(f"{DELIVERY}/{name}"):
        urllib.request.urlretrieve(url, f"{DELIVERY}/{name}")
!ls -la {DELIVERY}

In [ ]:
!brats-uncertainty verify-checksums --sums {DELIVERY}/RSNA-ASNR-MICCAI-BraTS-2021.sums --root {DELIVERY} --select RSNA-ASNR-MICCAI-BraTS-2021/BraTS2021_TrainingSet --out {EXPORT}/B2_provider_checksums.json

In [ ]:
ROUTE = "Direct official TCIA access into a private, access-restricted computational environment"
ACQUIRED_BY = ""  # owner name
VERSION = "Version 1 (2023/08/25)"
assert ACQUIRED_BY
common = (
    f"--adapter local-import --delivered {DELIVERY} "
    f'--dataset-version "{VERSION}" --route "{ROUTE}" '
    f"--storage-root {WORK}/raw "
    f'--storage-label "private compute session storage" '
    f'--acquired-by "{ACQUIRED_BY}" --out {EXPORT}/B2.json'
)
print(common)
!brats-uncertainty acquire {common}   # dry run

In [ ]:
!brats-uncertainty acquire {common} --execute

## Hand-over
Download `export/` (`B2.json`, `B2_provider_checksums.json`, probe JSON). The owner reviews them, commits them as evidence and moves B2 with `gate-transition` in the owner's checkout. Later sessions re-download and must pass `brats-uncertainty verify-inventory --record B2.json --root <raw>` before using the data. B3-B6 then follow DATA_PROVENANCE.md §3.